In [ ]:
import nipype
import nipype.pipeline as pe
import nipype.interfaces.io as io
import nipype.interfaces.fsl as fsl

# Working Directory
WorkingDirectory = "~/Porcupipelines/ThisStudy"

# Flexibly collect data from disk to feed into workflows.
selectfiles = pe.Node(io.SelectFiles(templates={
    'anat': 'data/ds001_R2.0.4/sub-{sub_id}/anat/sub-{sub_id}_T1w.nii.gz',
    'func': 'data/ds001_R2.0.4/sub-{sub_id}/func/sub-{sub_id}_task-balloonanalogrisktask_run-01_bold.nii.gz'
}), name='selectfiles')
selectfiles.inputs.base_directory = 'data'
selectfiles.iterables = ('sub_id', ['01', '02', '03'])

# Node for fslroi
extractroi = pe.Node(interface=fsl.ExtractROI(t_min=0, t_size=1), name='extractroi')

# Node for BET skull stripping
skullstrip = pe.Node(interface=fsl.BET(mask=True), name='skullstrip')

# Node for FLIRT registration
flirt = pe.Node(interface=fsl.FLIRT(), name='flirt')

# DataSink to store outputs
datasink = pe.Node(interface=io.DataSink(base_directory='results'), name='datasink')
# Use substitutions to define how outputs are stored in the DataSink
substitutions = [('_subject_id_', '')]  # Remove 'subject_id' from folder structure in DataSink
datasink.inputs.substitutions = substitutions

# Create a workflow to connect all those nodes
analysisflow = nipype.Workflow('MyWorkflow')

# Connect nodes
analysisflow.connect([
    (selectfiles, extractroi, [('func', 'in_file')]),
    (selectfiles, skullstrip, [('anat', 'in_file')]),
    (skullstrip, flirt, [('out_file', 'in_file')]),
    (extractroi, flirt, [('roi_file', 'reference')]),
    (flirt, datasink, [('out_file', 'registration')]),
    (skullstrip, datasink, [('out_file', 'skullstrip')]),
])

# Generate the workflow graph
try:
    analysisflow.write_graph(graph2use='flat', format='png', simple_form=False)
except RuntimeError as e:
    print(f"RuntimeError during graph generation: {e}")

# Run the workflow
plugin = 'MultiProc'  # adjust your desired plugin here ('MultiProc', 'Condor', etc.)
plugin_args = {'n_procs': 1}  # adjust to your number of cores
try:
    analysisflow.run(plugin=plugin, plugin_args=plugin_args)
except RuntimeError as e:
    print(f"RuntimeError during workflow execution: {e}")
    for node in analysisflow._get_all_nodes():
        if node.result and node.result.runtime and node.result.runtime.stderr:
            print(f"Node: {node.name}")
            print(node.result.runtime.stderr)


In [ ]:
!pip install nilearn

In [ ]:
import nibabel as nib
from nilearn import plotting

# File paths
registered_file = 'results/registration/_sub_id_01/sub-01_T1w_brain_flirt.nii.gz'
skullstripped_file = 'results/skullstrip/_sub_id_01/sub-01_T1w_brain.nii.gz'

# Load the images using nibabel
registered_img = nib.load(registered_file)
skullstripped_img = nib.load(skullstripped_file)

# Display interactive whole brain images using nilearn and save as HTML
plot_registered = plotting.view_img(registered_img, title='Registered Image - Raw Data', threshold=0, colorbar=False, cut_coords=(10, 10, 10), cmap="gray")
plot_registered.save_as_html('data/brain_plots/registered_image_raw.html')

plot_skullstripped = plotting.view_img(skullstripped_img, title='Skull Stripped Image - Interactive View', threshold=None, colorbar=True, cut_coords=None)
plot_skullstripped.save_as_html('data/brain_plots/skull_stripped_image.html')

In [ ]:
import os
from nilearn import datasets, image, plotting

# Define paths
roi_dir = 'data/ROI'
os.makedirs(roi_dir, exist_ok=True)

# Download the Harvard-Oxford atlas
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')

# Define labels for the language network and prefrontal cortex
language_labels = ['Superior Temporal Gyrus, posterior division',
                   'Middle Temporal Gyrus, posterior division',
                   'Inferior Frontal Gyrus, pars opercularis']

prefrontal_labels = ['Frontal Pole',
                     'Superior Frontal Gyrus',
                     'Middle Frontal Gyrus']

# Function to create ROI mask from atlas
def create_roi_mask(atlas, labels, output_path):
    labels_img = image.load_img(atlas.maps)
    labels_data = labels_img.get_fdata()

    roi_mask = sum((labels_data == i).astype(int)
                   for i, label in enumerate(atlas.labels) if label in labels)
    roi_mask_img = image.new_img_like(labels_img, roi_mask)
    roi_mask_img.to_filename(output_path)

# Create and save language network mask
language_mask_path = os.path.join(roi_dir, 'language_network.nii.gz')
create_roi_mask(atlas, language_labels, language_mask_path)

# Create and save prefrontal cortex mask
prefrontal_mask_path = os.path.join(roi_dir, 'prefrontal_cortex.nii.gz')
create_roi_mask(atlas, prefrontal_labels, prefrontal_mask_path)

# Plot the ROIs
plotting.plot_roi(language_mask_path, title='Language Network Mask')
plotting.plot_roi(prefrontal_mask_path, title='Prefrontal Cortex Mask')
plotting.show()


In [ ]:
import os

data_dir = 'data/ds000051'
subject_id = '01'
subject_func_dir = os.path.join(data_dir, f'sub-{subject_id}', 'func')

print(f"Listing files in: {subject_func_dir}")
print(os.listdir(subject_func_dir))


In [ ]:
import os
import nipype.interfaces.utility as util
import nipype.interfaces.io as nio
import nipype.pipeline.engine as pe
from nipype.algorithms.confounds import TSNR
from nipype.interfaces.fsl import FLIRT, ImageMeants

# Define base directory and subject list
base_dir = 'data/'
data_dir = os.path.join(base_dir, 'data')
output_dir = os.path.join(base_dir, 'output')

# Define the subject list
subject_list = ['sub-01', 'sub-02']  # Example subject list

# Function to check file existence
def check_file(file):
    import os
    if not os.path.exists(file):
        raise FileNotFoundError(f"File not found: {file}")
    return file

# Function to extract mean time series
def extract_mean_timeseries(func_file, mask_file):
    from nipype.interfaces.fsl import ImageMeants
    meants = ImageMeants()
    meants.inputs.in_file = func_file
    meants.inputs.mask = mask_file
    result = meants.run()
    return result.outputs.out_file

# Create the workflow
wf = pe.Workflow(name='functional_connectivity_analysis', base_dir=output_dir)

# Node to select files
templates = {'func': 'sub-{subject_id}/func/*_bold.nii.gz',
             'anat': 'sub-{subject_id}/anat/*_T1w.nii.gz'}
selectfiles = pe.Node(nio.SelectFiles(templates, base_directory=data_dir), name='selectfiles')
selectfiles.iterables = [('subject_id', subject_list)]

# Node to compute tSNR
compute_tsnr = pe.Node(TSNR(), name='compute_tsnr')

# Node to register anatomical image
reg_anat = pe.Node(FLIRT(out_file='reg_anat.nii.gz'), name='reg_anat')

# Node to check file existence
check_file_node = pe.Node(util.Function(input_names=['file'], output_names=['file'], function=check_file), name='check_file')

# Node to extract mean ROI time series
extract_mean_roi = pe.Node(util.Function(input_names=['func_file', 'mask_file'], output_names=['out_file'], function=extract_mean_timeseries), name='extract_mean_roi')

# Connect the nodes
wf.connect([
    (selectfiles, compute_tsnr, [('func', 'in_file')]),
    (selectfiles, reg_anat, [('anat', 'in_file')]),
    (reg_anat, check_file_node, [('out_file', 'file')]),
    (compute_tsnr, extract_mean_roi, [('tsnr_file', 'func_file')]),
    (check_file_node, extract_mean_roi, [('file', 'mask_file')])
])

# Execute the workflow
if __name__ == '__main__':
    wf.run('MultiProc', plugin_args={'n_procs': 4})

In [ ]:
import os
from nipype import Workflow, Node, MapNode
from nipype.interfaces import fsl, utility, io

# Paths and directories
base_dir = 'data/ds000051'
data_dir = os.path.join(base_dir, 'sub-{subject_id}', 'func')
output_dir = os.path.join(base_dir, 'fmri_preprocessing')

# Subject and run identifiers
subject_list = ['01', '02']  # List of subjects
run_list = ['01']  # List of runs

# Initialize Workflow
preproc_wf = Workflow(name='fmri_preprocessing', base_dir=output_dir)

# Node to select files
selectfiles = Node(io.SelectFiles(templates={'func': os.path.join(data_dir, 'sub-{subject_id}_task-abstractconcretejudgment_run-{run_id}_bold.nii.gz')}), 
                   name='selectfiles')

# Node for motion correction using MCFLIRT
mcflirt = MapNode(fsl.MCFLIRT(mean_vol=True, save_plots=True), 
                  name='mcflirt', 
                  iterfield=['in_file'])

# Node for slice timing correction using Slicetimer
slicetimer = MapNode(fsl.SliceTimer(time_repetition=2.5), 
                     name='slicetimer', 
                     iterfield=['in_file'])

# Node for spatial smoothing
smooth = MapNode(fsl.Smooth(fwhm=5), 
                 name='smooth', 
                 iterfield=['in_file'])

# DataSink node to store outputs
datasink = Node(io.DataSink(base_directory=output_dir), 
                name='datasink')

# Connect the nodes
preproc_wf.connect([
    (selectfiles, mcflirt, [('func', 'in_file')]),
    (mcflirt, slicetimer, [('out_file', 'in_file')]),
    (slicetimer, smooth, [('slice_time_corrected_file', 'in_file')]),
    (smooth, datasink, [('smoothed_file', 'preprocessed_func')]),
])

# Iterate over subjects and runs
subject_infosource = Node(utility.IdentityInterface(fields=['subject_id', 'run_id']), 
                          name='subject_infosource')
subject_infosource.iterables = [('subject_id', subject_list), ('run_id', run_list)]

preproc_wf.connect(subject_infosource, 'subject_id', selectfiles, 'subject_id')
preproc_wf.connect(subject_infosource, 'run_id', selectfiles, 'run_id')

# Run the workflow
preproc_wf.run('MultiProc', plugin_args={'n_procs': 4})